# Clase 13 — Regresión Lineal: Tu Primer Modelo Predictivo

**Diplomado en Data Science Aplicada con Python para la Toma de Decisiones**  
Arca Continental Ecuador | UDLA  
*8 de Abril, 2026*

---

## Objetivos de hoy

1. Dar el salto de **describir** datos a **predecir** con ellos
2. Entender la regresión lineal **simple** ($\hat{y} = \beta_0 + \beta_1 x$)
3. Escalar a regresión lineal **múltiple** y aprender a interpretar coeficientes en lenguaje de negocio
4. Aplicar **encoding** (clase 12) y **estandarización** (z-score reaparece) en un pipeline real
5. Aprender el flujo `train_test_split → fit → predict → evaluar` que usaremos para todo lo que viene

**Dataset**: `insurance.csv` — 1338 personas, 7 columnas, sin datos faltantes.

## 0. Imports y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

# Modelos y herramientas
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print("Listo.")

In [ ]:
# Cargamos directamente desde GitHub (no necesitas descargar nada)
URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
df = pd.read_csv(URL)

print(f"{df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

In [ ]:
# Verificacion rapida: tipos y NaN
df.info()
print(f"\nValores faltantes totales: {df.isna().sum().sum()}")

**Observaciones:**
- 1338 filas, 7 columnas, **sin NaN**.
- 3 numéricas (`age`, `bmi`, `children`), 3 categóricas (`sex`, `smoker`, `region`), 1 target (`charges`).
- `charges` es lo que queremos predecir.

In [ ]:
# Estadistica basica del target
df["charges"].describe()

In [ ]:
# Histograma + KDE de charges (lo que vimos en clase 9)
sns.histplot(df["charges"], bins=40, kde=True, color="#C82B40")
plt.title("Distribución de charges")
plt.xlabel("Charges (USD)")
plt.show()

Y un boxplot del mismo `charges` para ver la mediana, los cuartiles y los outliers de un vistazo:

In [ ]:
sns.boxplot(x=df["charges"], color="#C82B40")
plt.title("Boxplot de charges")
plt.show()

**Lo que vemos:** la distribución de `charges` está muy sesgada a la derecha. La mayoría paga menos de \$10k, pero hay una cola larga de personas que pagan \$30k–\$60k. Esa cola probablemente sea de fumadores. **Lo confirmaremos en breve.**

### 0.1 Una pista visual del culpable

Antes de modelar, separemos `charges` por **fumador / no fumador** con un KDE. Es la gráfica más reveladora de todo el dataset:

**Algo nuevo: el parámetro `hue=`**

Hasta ahora hemos hecho una gráfica por variable. Pero seaborn permite **separar por grupos** automáticamente con el parámetro `hue=`. Le pasas el nombre de una columna **categórica** (aquí `smoker`) y dibuja una curva distinta para cada categoría, con su propio color.

Es la forma más simple de comparar distribuciones entre grupos.

In [ ]:
# hue="smoker" hace que seaborn dibuje una curva por categoria
sns.kdeplot(data=df, x="charges", hue="smoker", fill=True)
plt.title("Distribución de charges según smoker")
plt.show()

**Lo que vemos a simple vista:**
- Los **no fumadores** (gris) están concentrados a la izquierda, casi todos por debajo de \$15k.
- Los **fumadores** (rojo) tienen una distribución completamente desplazada a la derecha, muchos por encima de \$30k.
- Las dos distribuciones **apenas se solapan**.

Esto explica por qué `smoker` será — sin sorpresa — la variable más importante del modelo.

---

## 1. Vocabulario nuevo: Target y Features

Antes de modelar, fijemos dos términos que vas a escuchar mil veces de aquí en adelante:

- **Target** (`y`): lo que quieres predecir. Aquí: `charges`.
- **Features** (`X`): lo que usas para predecir. Aquí: las otras 6 columnas.

Convención universal en ML: `X` mayúscula (matriz), `y` minúscula (vector).

In [ ]:
y = df["charges"]
X = df.drop(columns=["charges"])

print(f"Target:   {y.name} (vector de {len(y)} valores)")
print(f"Features: {list(X.columns)}")

---

## 2. Regresión Lineal Simple — `charges` en función de `age`

Empezamos con una sola feature para construir intuición.

### 2.1 Visualización: ¿hay una relación?

In [ ]:
# Scatter clasico: cada punto es una persona
sns.scatterplot(data=df, x="age", y="charges", alpha=0.4, color="#6B1525")
plt.title("charges vs age")
plt.show()

**Observación clave:** se ve una tendencia ascendente (a más edad, más costo), pero hay **tres bandas paralelas** muy marcadas. Adelantamos: esas tres bandas son fumadores vs no-fumadores (lo veremos al final). Por ahora, ajustemos una recta a todo.

### 2.2 Ajustar la recta con sklearn

### 2.1.1 Misma gráfica, coloreada por smoker

Las "tres bandas" que se ven se vuelven obvias cuando coloreamos por fumador:

In [ ]:
# El mismo scatter, ahora separado por smoker con hue=
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=0.6)
plt.title("charges vs age, separado por smoker")
plt.show()

**Ahora todo tiene sentido:**
- Los **no fumadores** (gris) forman una banda inferior, con una clara tendencia ascendente con la edad.
- Los **fumadores** (rojo) forman dos bandas superiores — la diferencia entre ellas probablemente sea por BMI.
- La regresión simple `charges ~ age` que vamos a entrenar **ignora esta separación** y dibuja una sola recta en el medio. Por eso su R² será tan bajo.

In [ ]:
# IMPORTANTE: X debe ser DataFrame (2D), no Series (1D)
X_simple = df[["age"]]   # doble corchete -> DataFrame
y = df["charges"]

modelo = LinearRegression()
modelo.fit(X_simple, y)

print(f"Intercepto (beta_0): {modelo.intercept_:.2f}")
print(f"Pendiente  (beta_1): {modelo.coef_[0]:.2f}")

### 2.3 Interpretación de los coeficientes

$$\hat{\text{charges}} = 3165.89 + 257.72 \cdot \text{age}$$

- **Intercepto (3165.89):** valor de `charges` cuando `age = 0`. Matemáticamente útil, pero **no tiene significado literal** (nadie tiene 0 años).
- **Pendiente (257.72):** **por cada año adicional de edad, el costo del seguro sube \$257.72**, en promedio.

Esa traducción coeficiente → frase de negocio es lo más importante de toda la clase.

In [ ]:
# Predecir el costo para 3 personas hipoteticas
nuevas_edades = pd.DataFrame({"age": [25, 40, 60]})
predicciones = modelo.predict(nuevas_edades)

for edad, pred in zip([25, 40, 60], predicciones):
    print(f"Edad {edad}: ${pred:,.2f}")

### 2.4 Visualizar la recta ajustada y los residuos

**`sns.regplot` ya lo vimos en la clase 11**: dibuja el scatter Y la recta de regresión que mejor se ajusta, todo en un solo comando. Es exactamente lo que necesitamos.

In [ ]:
# regplot: scatter + recta de regresion (lo que vimos en clase 11)
sns.regplot(data=df, x="age", y="charges",
            scatter_kws={"alpha": 0.4, "color": "#6B1525"},
            line_kws={"color": "#C82B40"})
plt.title("Regresión lineal simple: charges ~ age")
plt.show()

La recta es claramente una **simplificación brutal**: hay mucha dispersión vertical alrededor de ella. Eso lo va a confirmar el R² que veremos a continuación.

### 2.5 ¿Cómo medimos qué tan bueno es el modelo?

Tenemos predicciones. Pero ¿cómo le ponemos un **número** a qué tan equivocado está el modelo?

**Paso 1: los residuos.** Para cada persona, la diferencia entre lo real y lo predicho:

$$\text{residuo}_i = y_i - \hat{y}_i$$

In [ ]:
y_pred = modelo.predict(X_simple)
residuos = y - y_pred

# Tomamos 40 personas al azar para no saturar el grafico
muestra = residuos.sample(40, random_state=7).reset_index(drop=True)

# Dibujamos cada residuo como un punto, y ponemos una linea en cero
sns.scatterplot(x=muestra.index, y=muestra, color="#C82B40")
plt.axhline(0, color="black")  # cero = prediccion perfecta
plt.title("Residuos: cuánto se equivocó el modelo en cada persona")
plt.xlabel("personas (40 al azar)")
plt.ylabel("residuo (USD)")
plt.show()

**Problema:** hay barras positivas y negativas. Si las sumamos directamente, **se cancelan**. Y eso no significa que el modelo sea perfecto — significa que su sesgo es cero, nada más.

Necesitamos **resumir los residuos en un solo número** sin que se cancelen. Hay 3 formas estándar:

#### 2.5.1 MSE — Error Cuadrático Medio

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Promedio del **cuadrado** de los residuos. ¿Por qué al cuadrado?

1. Para que **positivos y negativos no se cancelen** (los cuadrados son siempre positivos).
2. Para **castigar más los errores grandes** (un error de \$1000 cuenta 100 veces más que uno de \$100, no 10 veces).

> **Dato importante:** MSE es exactamente lo que el modelo OLS está **minimizando** cuando lo entrenas. No es una métrica nueva mágica — es la cosa misma que `LinearRegression()` está optimizando bajo el capó.

In [ ]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y, y_pred)
print(f"MSE: {mse:,.0f}")
print("Unidad: USD al cuadrado (raro de leer)")

**Problema del MSE:** está en **dólares al cuadrado**. Nadie sabe leer "132 millones de dólares²". Lo arreglamos sacando la raíz cuadrada → **RMSE**.

#### 2.5.2 RMSE — Raíz del MSE

$$\text{RMSE} = \sqrt{\text{MSE}}$$

Vuelve a estar en dólares. Sigue castigando más los errores grandes (porque viene del cuadrado).

In [ ]:
rmse = np.sqrt(mse)
print(f"RMSE: ${rmse:,.0f}")

**Frase de negocio:** "el modelo se equivoca, en promedio, en \$11,510 al predecir el costo del seguro de una persona".

#### 2.5.3 MAE — Error Absoluto Medio

$$\text{MAE} = \frac{1}{n}\sum |y_i - \hat{y}_i|$$

Alternativa al RMSE: en vez de elevar al cuadrado, tomamos el **valor absoluto**. Más fácil de explicar, pero matemáticamente menos elegante (la función |·| no es derivable en 0).

In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y, y_pred)
print(f"MAE: ${mae:,.0f}")

**Comparación**:
- **MAE = \$9,090** → trata todos los errores por igual.
- **RMSE = \$11,510** → es mayor que MAE porque castiga más los errores grandes (los outliers contribuyen más).

Cuando RMSE >> MAE, sabes que tu modelo tiene **algunos errores muy grandes** entre muchos pequeños. Cuando son parecidos, los errores son uniformes.

#### 2.5.4 R² — la comparación con el modelo tonto

OK, ya sabemos que MAE = \$9,090. Pero **¿eso es mucho o poco?** Necesitamos un **punto de comparación**.

El más tonto posible: **predecir siempre la media** de `charges`. Ese es el peor modelo razonable — ignora todas las features y siempre dice lo mismo.

In [ ]:
# Modelo TONTO: predecir siempre la media
media = y.mean()
y_pred_tonto = np.full_like(y, media)

mse_tonto = mean_squared_error(y, y_pred_tonto)
mae_tonto = mean_absolute_error(y, y_pred_tonto)

print(f"Modelo tonto (predecir siempre la media de ${media:,.0f}):")
print(f"  MSE: {mse_tonto:,.0f}")
print(f"  MAE: ${mae_tonto:,.0f}")
print()
print(f"Nuestro modelo (charges ~ age):")
print(f"  MSE: {mse:,.0f}")
print(f"  MAE: ${mae:,.0f}")

**Visualicémoslo:** dos rectas en el mismo scatter, una horizontal (la media) y otra inclinada (la regresión). Las líneas grises/rojas son los residuos de cada una.

In [ ]:
# MODELO TONTO: predecir siempre la media
sns.scatterplot(data=df, x="age", y="charges", alpha=0.4, color="#6B1525")
plt.axhline(media, color="#888", linewidth=2.5)  # linea horizontal en la media
plt.title("Modelo TONTO: predecir siempre la media")
plt.show()

Y ahora el **mismo scatter**, pero con la recta de regresión real encima. Visualmente, los puntos están más cerca de la recta inclinada que de la línea horizontal — esa diferencia es lo que mide R².

In [ ]:
# MODELO DE REGRESION: usamos regplot otra vez
sns.regplot(data=df, x="age", y="charges",
            scatter_kws={"alpha": 0.4, "color": "#6B1525"},
            line_kws={"color": "#C82B40"})
plt.title("Modelo de REGRESIÓN lineal")
plt.show()

**R² es exactamente esa comparación:**

$$R^2 = 1 - \frac{\text{MSE del modelo}}{\text{MSE del modelo tonto}}$$

- **R² = 0**: tu modelo es **igual de malo** que predecir la media (no aporta nada).
- **R² = 1**: tu modelo es **perfecto** (MSE = 0, residuos = 0).
- **R² = 0.7**: tu modelo eliminó el **70 %** del error que tendría el modelo tonto.

Ahora R² ya no es abstracto — es literalmente "qué tan mejor soy que la opción más tonta".

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y, y_pred)
print(f"R^2: {r2:.4f}")
print()

# Calculo manual para verificar
r2_manual = 1 - (mse / mse_tonto)
print(f"R^2 manual = 1 - mse/mse_tonto = {r2_manual:.4f}")

**Lectura completa de nuestro modelo simple `charges ~ age`:**

| Métrica | Valor | Frase de negocio |
|---|---|---|
| **MAE** | \$9,090 | "en promedio nos equivocamos en \$9k" |
| **RMSE** | \$11,510 | "cuando se equivoca feo, falla por más de \$11k" |
| **R²** | 0.089 | "solo eliminamos el 9 % del error del modelo tonto" |

**Las 3 métricas dicen lo mismo con palabras distintas: el modelo es muy malo.** La edad sola no alcanza para predecir el seguro.

¿La solución? **Agregar más variables.** Eso es regresión múltiple.

### 2.5.5 Tabla de referencia: cómo leer cada métrica

| Métrica | Unidad | Rango | Cómo se lee |
|---|---|---|---|
| **MSE** | USD² | [0, ∞) | "Error cuadrático promedio". Difícil de leer en negocio. Es la cantidad que el modelo **minimiza**. |
| **RMSE** | USD | [0, ∞) | "Cuando el modelo se equivoca *feo*, falla por ≈ \$X". Castiga más los errores grandes. |
| **MAE** | USD | [0, ∞) | "En promedio, el modelo se equivoca en \$X". Trata todos los errores por igual. La más fácil de explicar. |
| **R²** | sin unidad | (−∞, 1] | "El modelo eliminó el X% del error que tendría predecir siempre la media". Score relativo. |

> **Comparación útil**: si **RMSE >> MAE**, hay errores grandes aislados (outliers en los residuos). Si **RMSE ≈ MAE**, los errores son uniformes.

### 2.5.6 ¿Cuándo usar cada una?

| Métrica | Cuándo SÍ usarla | Cuándo NO |
|---|---|---|
| **MAE** | Reportes a no técnicos. Cuando todos los errores deben pesar igual. Cuando hay outliers que no quieres que dominen. | Cuando errores grandes son *mucho* más caros que los pequeños. |
| **RMSE** | Cuando errores grandes son críticos (medicina, finanzas, predicción de demanda). Para presentar resultados técnicos. | Cuando hay outliers extremos que distorsionan: lo van a inflar. |
| **MSE** | Como *loss function* interna. En papers académicos. Para optimizar el modelo (es lo que OLS minimiza). | Para reportar a humanos: las unidades al cuadrado son ilegibles. |
| **R²** | Para **comparar modelos** sobre el mismo dataset. Para resúmenes ejecutivos ("el modelo explica el 78%"). | Para comparar modelos en **datasets distintos**: un R² de 0.4 puede ser excelente en un dominio y malísimo en otro. |

> **Combo recomendado para reportar**: **MAE** (en USD, fácil de explicar) **+ R²** (score sin unidades para comparar modelos). Con esos dos cubren ejecutivos y científicos.

In [ ]:
df_enc = pd.get_dummies(
    df,
    columns=["sex", "smoker", "region"],
    drop_first=True,
    dtype=int
)
df_enc.head()

In [ ]:
print("Columnas resultantes:")
for c in df_enc.columns:
    print(f"  {c}")

Notas:
- `sex_male = 1` significa hombre, `0` mujer (la categoría de referencia es `female`).
- `smoker_yes = 1` fumador, `0` no fumador.
- `region` se convirtió en 3 columnas (la 4ª, `northeast`, es la referencia).

### 3.2 Entrenar el modelo múltiple

In [ ]:
X = df_enc.drop(columns=["charges"])
y = df_enc["charges"]

modelo_m = LinearRegression()
modelo_m.fit(X, y)

print(f"R^2: {modelo_m.score(X, y):.4f}\n")

# Tabla bonita de coeficientes
coefs = pd.DataFrame({
    "feature": X.columns,
    "coeficiente": modelo_m.coef_
}).sort_values("coeficiente", key=abs, ascending=False)
print(coefs.to_string(index=False))
print(f"\n(intercepto: {modelo_m.intercept_:.2f})")

### 3.3 Lectura de resultados

**El R² pasó de 0.09 a 0.75**: una mejora brutal. Pero el coeficiente que más debe llamarte la atención es:

$$\beta_{\text{smoker\_yes}} \approx +23{,}848$$

**Lectura literal:** ser fumador suma **\$23,848** al costo del seguro, *manteniendo* edad, BMI, sexo, hijos y región constantes.

**Lectura de negocio:** dos personas idénticas en todo lo demás, pero una fuma — pagarán **\$24,000 de diferencia anual**.

Eso es lo que significa el coeficiente de una **dummy** en regresión: el "salto" que produce esa categoría respecto a su categoría de referencia.

### 3.4 "Manteniendo lo demás constante" — ¿y si quito una variable?

In [ ]:
# Modelo SIN smoker
X_sin_smoker = X.drop(columns=["smoker_yes"])
m_sin = LinearRegression().fit(X_sin_smoker, y)

print(f"R^2 sin smoker: {m_sin.score(X_sin_smoker, y):.4f}")
print(f"R^2 con smoker: {modelo_m.score(X, y):.4f}")
print()
print("Coeficiente de age:")
print(f"  con smoker:  {modelo_m.coef_[0]:.2f}")
print(f"  sin smoker:  {m_sin.coef_[0]:.2f}")

**Lo que pasó:**
- El R² cae de 0.75 a 0.12 cuando quitamos `smoker`. **Una sola variable explica más que las otras 7 juntas.**
- El coeficiente de `age` cambia ligeramente: en presencia o ausencia de `smoker`, el "efecto" estimado de la edad cambia. Esa es la razón por la que la frase "manteniendo lo demás constante" importa: el coeficiente depende del contexto.

---

## 4. Estandarización — comparar los coeficientes

### 4.1 La trampa: ¿cuál variable importa más?

Mirando los coeficientes del modelo múltiple:

| Variable     | Coeficiente | Unidad           |
|--------------|-------------|------------------|
| age          | +257        | por año          |
| bmi          | +339        | por unidad BMI   |
| children     | +476        | por hijo         |
| smoker_yes   | +23 848     | binario (0/1)    |

**No puedes comparar estos números directamente** porque están en unidades distintas. Necesitamos ponerlos todos en la misma escala.

### 4.2 El z-score reaparece (clases 9 y 10)

$$z = \frac{x - \mu}{\sigma}$$

Es **exactamente** la misma fórmula que usamos para detectar outliers en clase 9. Pero ahora la usamos como **preprocesamiento de features**: cualquier variable queda con media 0 y desviación 1.

**`sklearn` la implementa como `StandardScaler`.**

**Veámoslo gráficamente.** Pongamos las 4 variables numéricas en el **mismo gráfico** con KDE:

In [ ]:
# Llamamos sns.kdeplot 4 veces, una por variable
# label= le da nombre a cada curva para la leyenda
sns.kdeplot(df["age"],      label="age",      fill=True)
sns.kdeplot(df["bmi"],      label="bmi",      fill=True)
sns.kdeplot(df["children"], label="children", fill=True)
sns.kdeplot(df["charges"],  label="charges",  fill=True)
plt.title("4 variables superpuestas en el mismo eje")
plt.xlabel("valor (en su unidad original)")
plt.legend()
plt.show()

**El problema salta a la vista:**
- `children` es prácticamente **invisible** (su rango es 0 a 5).
- `age` y `bmi` se ven como picos delgados a la izquierda.
- `charges` ocupa casi todo el gráfico porque va de 0 a 64 000.

**No podemos compararlas en este estado.** Necesitamos llevarlas todas a la misma escala.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Verificar: media ~0, desviacion ~1
print(f"Medias:       {X_scaled.mean(axis=0).round(2)}")
print(f"Desviaciones: {X_scaled.std(axis=0).round(2)}")

### 4.2.1 La transformación, visualizada en una sola variable

Tomemos solo `age` y veamos qué le pasa al estandarizarla:

In [ ]:
mu    = df["age"].mean()
sigma = df["age"].std()

# ANTES: age en sus unidades originales
sns.kdeplot(df["age"], fill=True, color="#C82B40")
plt.axvline(mu,         color="black", linestyle="--", label=f"media = {mu:.1f}")
plt.axvline(mu - sigma, color="gray",  linestyle=":")
plt.axvline(mu + sigma, color="gray",  linestyle=":", label=f"± 1 σ = {sigma:.1f}")
plt.title("ANTES: age en años")
plt.xlabel("años")
plt.legend()
plt.show()

Ahora aplicamos $z = (x - \mu) / \sigma$ y volvemos a graficar. **La forma es idéntica**, solo cambia el eje:

In [ ]:
# DESPUES: age estandarizada
zage = (df["age"] - mu) / sigma

sns.kdeplot(zage, fill=True, color="#6B1525")
plt.axvline(0,  color="black", linestyle="--", label="media = 0")
plt.axvline(-1, color="gray",  linestyle=":")
plt.axvline(1,  color="gray",  linestyle=":", label="± 1 σ = 1")
plt.title("DESPUÉS: age estandarizada")
plt.xlabel("desviaciones estándar")
plt.legend()
plt.show()

**Lo importante:** la **forma** de la distribución es idéntica. Lo único que cambia es **el eje**. Antes la media estaba en 39.2 años, ahora está en 0. Antes una "vuelta" valía 1 año, ahora vale 1 desviación estándar.

**No se pierde información.** Solo se reescala.

In [ ]:
# Re-entrenar el modelo con X estandarizado
modelo_std = LinearRegression().fit(X_scaled, y)

coefs_std = pd.DataFrame({
    "feature": X.columns,
    "coef_estandarizado": modelo_std.coef_
}).sort_values("coef_estandarizado", key=abs, ascending=False)

print(coefs_std.to_string(index=False))

### 4.2.2 Las 4 variables, ahora superpuestas en el mismo eje

Volvemos a hacer el gráfico de antes — pero con las features estandarizadas:

In [ ]:
# Estandarizamos solo las 4 numericas para ilustrar
df_num_std = pd.DataFrame(
    StandardScaler().fit_transform(df[["age", "bmi", "children", "charges"]]),
    columns=["age", "bmi", "children", "charges"]
)

# Mismas 4 KDEs, ahora con los datos estandarizados
sns.kdeplot(df_num_std["age"],      label="age",      fill=True)
sns.kdeplot(df_num_std["bmi"],      label="bmi",      fill=True)
sns.kdeplot(df_num_std["children"], label="children", fill=True)
sns.kdeplot(df_num_std["charges"],  label="charges",  fill=True)
plt.axvline(0, color="black", linestyle="--")
plt.title("Las mismas 4 variables, después de estandarizar")
plt.xlabel("valor estandarizado")
plt.legend()
plt.show()

**Compara con la gráfica anterior:** ahora las cuatro están **visibles**, todas centradas en 0, todas en el rango \[-3, 3\]. **Las formas se conservan** (asimetrías, picos, colas) — la transformación es 100 % reversible y no destruye información.

### 4.2.3 Bar chart: ranking de importancia

Con los coeficientes estandarizados, podemos hacer una gráfica de barras que muestra de un vistazo qué variable importa más:

**Algo nuevo: `sns.barplot`**

Para gráficas de barras seaborn ofrece `sns.barplot`. Le pasas dos columnas de un DataFrame (x y y) y dibuja una barra por cada fila. Aquí lo usamos para mostrar el ranking de coeficientes:

In [ ]:
# Ordenamos por magnitud (absoluto) de mayor a menor
coefs_ord = coefs_std.copy()
coefs_ord["abs"] = coefs_ord["coef_estandarizado"].abs()
coefs_ord = coefs_ord.sort_values("abs", ascending=False)

# barplot horizontal: x = numero, y = categoria
sns.barplot(data=coefs_ord, x="coef_estandarizado", y="feature", color="#C82B40")
plt.axvline(0, color="black")
plt.title("Importancia de cada variable (coeficientes estandarizados)")
plt.xlabel("USD por desviación estándar")
plt.show()

### 4.3 Ahora sí: ranking de importancia

Con todas las variables en la misma escala, el ranking es claro:

1. **smoker_yes** — el más importante por mucho
2. **age** — segundo, ~3× menor
3. **bmi** — tercero
4. **children** — cuarto, mucho menor
5. Las regiones y el sexo: prácticamente irrelevantes

### 4.4 ⚠️ Importante: las predicciones NO cambian

In [ ]:
# Mismo modelo, dos versiones (sin escalar / escalado)
pred_sin = modelo_m.predict(X[:5])
pred_con = modelo_std.predict(X_scaled[:5])

pd.DataFrame({
    "pred_sin_escalar": pred_sin,
    "pred_escalado":    pred_con,
    "diferencia":       pred_sin - pred_con
})

**Las predicciones son idénticas** (la diferencia es ruido numérico). Lo único que cambia al escalar son los **coeficientes** y su interpretación. Estandarizar **no mejora ni empeora** la regresión lineal pura — solo permite comparar la importancia de las variables.

> Esto cambia con otros modelos (KNN, SVM, regresión regularizada, redes neuronales): ahí estandarizar **sí** afecta las predicciones. Lo veremos en el Módulo 4.

### 4.5 ⚠️ Data leakage: cómo NO hacerlo

La **regla de oro**: `fit` solo con los datos de entrenamiento. `transform` con los parámetros aprendidos en train, aplicado a test. Veremos esto en el siguiente bloque.

---

## 5. Train / Test Split — ¿cómo sé si mi modelo es bueno?

### 5.1 La pregunta provocadora

Acabamos de ver R² = 0.75. Pero **lo calculamos sobre los mismos datos con los que entrenamos**. Es como darte un examen con las mismas preguntas que estudiaste — vas a sacar buena nota, pero no demuestra nada.

**Lo correcto**: separar los datos *antes* de entrenar. Una parte para enseñar (train), otra escondida para examinar (test).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test
    random_state=42     # reproducibilidad
)
print(f"Train: {X_train.shape[0]} filas")
print(f"Test:  {X_test.shape[0]} filas")

### 5.2 Pipeline correcto: estandarizar, entrenar, evaluar

In [ ]:
# Paso 1: estandarizar SOLO con train, aplicar a ambos
scaler = StandardScaler()
scaler.fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

# Paso 2: entrenar SOLO con train
modelo = LinearRegression()
modelo.fit(X_train_s, y_train)

# Paso 3: predecir en ambos
y_pred_train = modelo.predict(X_train_s)
y_pred_test  = modelo.predict(X_test_s)

# Paso 4: evaluar
print(f"R^2 train: {r2_score(y_train, y_pred_train):.4f}")
print(f"R^2 test : {r2_score(y_test,  y_pred_test):.4f}")

**R² train ≈ R² test**: el modelo generaliza bien. No hay overfitting.

> **Si vieras R² train = 0.99 y R² test = 0.40**, eso sería **overfitting**: el modelo memorizó en lugar de aprender. La regresión lineal es relativamente robusta a esto cuando hay pocas variables — pero modelos más complejos del Módulo 4 sí son muy propensos.

### 5.3 Métricas en unidades de negocio

In [ ]:
mae  = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2   = r2_score(y_test, y_pred_test)

print(f"MAE  test: ${mae:,.2f}")
print(f"RMSE test: ${rmse:,.2f}")
print(f"R^2  test: {r2:.4f}")

**Frase ejecutiva**: "En promedio, el modelo se equivoca en **\$4,181** al predecir el costo del seguro. RMSE = \$5,796 (que es mayor porque castiga más los errores grandes)."

| Métrica | Cuándo usarla |
|---|---|
| **MAE** | Reportar a no-técnicos. Está en las mismas unidades del target. |
| **RMSE** | Cuando los errores grandes son más costosos que los chicos. |
| **R²** | Comparar modelos entre sí. Pero **no** para entender el error en \$. |

### 5.4 Visualizar predicciones vs reales

In [ ]:
# Predicho vs real: si el modelo fuera perfecto, todos los puntos estarian en y=x
sns.scatterplot(x=y_test, y=y_pred_test, alpha=0.5, color="#6B1525")
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         color="#C82B40", linestyle="--")
plt.xlabel("Real")
plt.ylabel("Predicho")
plt.title("Predicho vs Real (test set)")
plt.show()

Y los **residuos vs la predicción**: si el modelo fuera bueno, los residuos deberían ser ruido aleatorio alrededor de 0 (sin patrones).

In [ ]:
residuos_test = y_test - y_pred_test

sns.scatterplot(x=y_pred_test, y=residuos_test, alpha=0.5, color="#6B1525")
plt.axhline(0, color="#C82B40", linestyle="--")
plt.xlabel("Predicción")
plt.ylabel("Residuo (real − predicho)")
plt.title("Residuos vs Predicción")
plt.show()

**Lo que vemos:**
- **Plot izquierdo**: la mayoría de puntos están cerca de la diagonal (buena predicción), pero hay un grupo de puntos que el modelo subestima (cerca de la esquina superior — son personas a las que predijo \$10k pero pagan \$40k+). Probablemente fumadores con condiciones extra.
- **Plot derecho**: los residuos NO son uniformes — se ven dos "bandas". Eso es señal de que la regresión lineal está dejando algo sin capturar. Modelos no lineales (que verás en Módulo 4) pueden mejorar esto.

### 5.5 ¿Cómo se distribuyen los residuos?

Un modelo bien ajustado debería tener residuos **centrados en 0** y aproximadamente **normales** (forma de campana). Veámoslo:

In [ ]:
residuos = y_test - y_pred_test

# KDE de los residuos: si el modelo es bueno, deberia ser una campana en 0
sns.kdeplot(residuos, fill=True, color="#6B1525")
plt.axvline(0, color="#C82B40", linestyle="--")
plt.title("Distribución de residuos en test")
plt.xlabel("residuo (real − predicho)")
plt.show()

print(f"Media de residuos: ${residuos.mean():,.2f}")
print(f"Std  de residuos: ${residuos.std():,.2f}")

**Lectura:**
- **Centro cerca de 0**: bien, el modelo no está sesgado consistentemente hacia arriba o abajo.
- **Cola larga a la derecha**: hay un grupo de personas a las que el modelo subestima fuerte (probablemente los fumadores con condiciones extra que vimos en el plot anterior).
- **No es perfectamente normal**: la regresión lineal está dejando estructura sin capturar. Modelos más flexibles (Módulo 4) pueden mejorar esto.

In [ ]:
# === PARTE A: charges ~ bmi ===
# Tu codigo aqui


In [ ]:
# === PARTE B: solo fumadores ===
df_fumadores = df[df["smoker"] == "yes"].copy()
print(f"Personas fumadoras: {len(df_fumadores)}")

# Tu codigo aqui


---

## Resumen — herramientas de hoy

| Concepto | Librería | Código clave |
|---|---|---|
| Modelo | sklearn | `LinearRegression().fit(X, y)` |
| Predicción | sklearn | `modelo.predict(X_new)` |
| R² | sklearn | `modelo.score(X, y)` |
| Encoding | pandas | `pd.get_dummies(drop_first=True)` |
| Estandarización | sklearn | `StandardScaler().fit_transform(X)` |
| Train/test split | sklearn | `train_test_split(test_size=0.2)` |
| MAE / RMSE | sklearn | `mean_absolute_error`, `mean_squared_error` |

## Próxima clase (viernes 10 de abril)

Hoy predijimos un **número** (\$). El viernes vamos a predecir un **sí/no**:
**Regresión logística — tu primer clasificador.** Cierre del Módulo 2.